In [1]:
!pip install langchain langchain-openai langchain-community langgraph python dotenv faiss-cpu pypdf

ERROR: Could not find a version that satisfies the requirement python (from versions: none)
ERROR: No matching distribution found for python


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition


In [ ]:
load_dotenv()

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [ ]:
loader = PyPDFLoader('splitting.pdf')
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chhunk_overlap = 100)
chunks = splitter.split_document(docs)

In [ ]:
len(chunks)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 3})

In [ ]:
# now from onwards this is using langgraph
# we will wrapp RAG into a tool
@tool
def rag_tool(query)

    """
    Retrieve only relevant information from the uploaded document.
    Use this tool when the user ask factual / conceptual questions
    that might be answered from the stored documents.
    """

    result = retriever.invoke(query)

    context = [docs.page_content for docs in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query':query,
        'context':context,
        'metadata':metadata
    }


In [ ]:
tools = [rag_tool]      # creating one list to add this tool
llm_with_tools = llm.bind_tools(tools)   # binding with llm, telling llm that he has a access of this tool

In [ ]:
# this is langgraph code
class ChatState(TypedDict):
  message : Annotated[list[BaseMessage], add_messages]


In [ ]:
# creating two nodes
# this is a chat node
def chat_node(state: ChatState):
  messages = state('messages')

  response = llm_with_tools.invoke(messages)

  return {'messages': [response]}



In [ ]:
# this is a tool node
tool_node = ToolNode(tools)

In [ ]:
#creating a graph
graph = StateGraph(ChatState)

#Adding two nodes
graph.add_node('chat_node', chat_node)
graph.add_node('tool_node', tool_node)

#defining edges
graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

# compiling
chatbot = graph.compile()

In [ ]:
#invoking chatbot

result = chatbot.invoke(
    {
        'message': [
            HumanMessage(content=('Using the pdf notes, What does the term "multitasking" refer to in operating systems?'))   # this is a ques related to that pdf
        ]
    }
)

In [ ]:
print(result['message'][-1].content)